# Phase 1 · Architecture 3 · Legal Prefix–Conv LM (2×T4)

In [1]:
%pip -q install "tokenizers>=0.20,<0.23" "huggingface_hub>=0.27,<1.0" ijson bert-score cloudpickle

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 8.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 56.2 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import shutil

In [3]:
import os, json, time, math, hashlib, pickle, zipfile, sys, socket, subprocess
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import ijson, cloudpickle
from huggingface_hub import HfFileSystem, hf_hub_download
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
import torch
from torch import nn
from torch.nn import functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint

START = time.monotonic()
OUT = Path('/kaggle/working/p1_arch3'); OUT.mkdir(parents=True, exist_ok=True)
DATA = OUT / 'data'; DATA.mkdir(exist_ok=True)
REPO = 'Exploration-Lab/CS-781-Capstone'
source = Path("/kaggle/input/datasets/harshvardhanvar/p1-v3-gpu/p1_arch3")
shutil.copytree(source, OUT, dirs_exist_ok=True)


SOURCES = ['acts', 'high_courts', 'supreme_court', 'tribunals']
WORD_BUDGET = [8000000, 36000000, 24000000, 12000000]
C = dict(vocab=32768, width=1024, layers=18, heads=16, ff=2816,
         seq=1536, prefix=1152, target=384, micro=2, accum=8,
         lr=3e-4, warmup=150, weight_decay=0.1, seed=781,
         session_hours=9., reserve_hours=1.)
RESUME = '/kaggle/input/datasets/harshvardhanvar/p1-v3-gpu/p1_arch3/latest.pt'
NDEV = 2
print(' | '.join(f'GPU {i}: {torch.cuda.get_device_name(i)}' for i in range(NDEV)))

GPU 0: Tesla T4 | GPU 1: Tesla T4


## Corpus and document split

In [4]:
from kaggle_secrets import UserSecretsClient
import os

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [5]:
fs = HfFileSystem()

def documents(source):
    path = f'datasets/{REPO}/Phase_1_Pretraining/Train/train_{source}.json'
    with fs.open(path, 'rb', block_size=8 << 20) as f:
        first = f.read(256).lstrip()[:1]; f.seek(0)
        rows = ijson.items(f, 'item') if first == b'[' else (v for _, v in ijson.kvitems(f, ''))
        for row in rows:
            yield row if isinstance(row, str) else row['text']

for source, budget in zip(SOURCES, WORD_BUDGET):
    used = held = 0
    with (DATA / f'{source}.jsonl').open('w') as tr, (DATA / f'{source}_val.jsonl').open('w') as va:
        for text in documents(source):
            text = ' '.join(text.split())
            split = int.from_bytes(hashlib.blake2b(text.encode(), digest_size=8).digest(), 'little') % 100
            words = text.count(' ') + 1
            if split == 0:
                if held < 250000:
                    va.write(json.dumps(text, ensure_ascii=False) + '\n'); held += words
            else:
                tr.write(json.dumps(text, ensure_ascii=False) + '\n'); used += words
            if used >= budget:
                break
    print(f'{source}: {used:,} training words, {held:,} validation words')

acts: 8,042,403 training words, 35,714 validation words
high_courts: 36,000,430 training words, 250,219 validation words
supreme_court: 24,000,283 training words, 150,190 validation words
tribunals: 12,005,708 training words, 124,902 validation words


In [6]:
tok = Tokenizer.from_file(str(Path(RESUME).parent / 'tokenizer.json'))
tok.save(str(OUT / 'tokenizer.json'))
PAD, BOS = tok.token_to_id('<pad>'), tok.token_to_id('<bos>')
C['vocab'] = math.ceil(tok.get_vocab_size() / 128) * 128
print(f'BPE vocabulary: {tok.get_vocab_size():,}')

BPE vocabulary: 32,768


## Training examples

In [7]:
rng = np.random.default_rng(C['seed'])
counts = {}; dev = []; clipped = np.zeros(2, dtype=np.int64); token_count = 0

def write_rows(pairs, stream):
    global token_count
    enc = tok.encode_batch([s for pair in pairs for s in pair])
    rows = np.full((len(pairs), C['seq'] + 2), PAD, dtype=np.uint16)
    for i in range(len(pairs)):
        a, b = enc[2*i].ids, enc[2*i+1].ids
        clipped[:] += [len(a) > C['prefix'] - 1, len(b) > C['target']]
        a = [BOS] + a[-(C['prefix'] - 1):]; b = b[:C['target']]
        rows[i, :len(a)+len(b)] = a + b
        rows[i, -2:] = [len(a), len(b)]
        token_count += len(a) + len(b)
    rows.tofile(stream)

for split in ['train', 'val']:
    count = 0
    with (DATA / f'{split}.bin').open('wb') as dst:
        pairs = []
        for source in SOURCES:
            name = source if split == 'train' else source + '_val'
            source_count = 0
            with (DATA / f'{name}.jsonl').open() as f:
                for line in f:
                    words = json.loads(line).split()
                    if len(words) < 640:
                        continue
                    offset = int(rng.integers(0, min(640, len(words) - 639)))
                    for start in range(offset, len(words) - 639, 640):
                        pair = (' '.join(words[start:start+512]), ' ' + ' '.join(words[start+512:start+640]))
                        pairs.append(pair); count += 1; source_count += 1
                        if split == 'val' and len([d for d in dev if d['source'] == source]) < 8:
                            dev.append(dict(source=source, prompt=pair[0], reference=pair[1].strip()))
                        if len(pairs) == 128:
                            write_rows(pairs, dst); pairs = []
                        if split == 'val':
                            break  # One validation window per document.
                    if split == 'val' and source_count >= 128:
                        break
        if pairs:
            write_rows(pairs, dst)
    counts[split] = count
(OUT / 'dev.json').write_text(json.dumps(dev))
(OUT / 'config.json').write_text(json.dumps(C, indent=2))
(OUT / 'data_meta.json').write_text(json.dumps(dict(counts=counts, tokens=int(token_count))))
train = np.memmap(DATA / 'train.bin', dtype=np.uint16, mode='r', shape=(counts['train'], C['seq']+2))
valid = np.memmap(DATA / 'val.bin', dtype=np.uint16, mode='r', shape=(counts['val'], C['seq']+2))
print(f"Rows: {counts} | stored tokens: {token_count:,}")
print(f'Clipped prompt/target rows: {clipped.tolist()}')

Rows: {'train': 77889, 'val': 121} | stored tokens: 61,467,405
Clipped prompt/target rows: [113, 29]


## Architecture 3: Legal Prefix–Conv LM

In [8]:
def norm(x, w):
    y = x.float()
    return (y * torch.rsqrt(y.square().mean(-1, keepdim=True) + 1e-6) * w).to(x.dtype)

def rope(x, cos, sin):
    a, b = x.float().chunk(2, -1)
    return torch.cat((a*cos-b*sin, b*cos+a*sin), -1).to(x.dtype)

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        d, f = c['width'], c['ff']
        self.h = c['heads']
        self.p = nn.ParameterDict({k: nn.Parameter(torch.empty(*shape)) for k, shape in dict(
            n1=(d,), n2=(d,), nc=(d,), conv=(3,d), gate=(d,),
            qkv=(d,3*d), out=(d,d), up=(d,2*f), down=(f,d)).items()})
        for name, w in self.p.items():
            if name in ('n1','n2','nc'): nn.init.ones_(w)
            elif name == 'gate': nn.init.zeros_(w)
            else: nn.init.normal_(w, std=0.02 / math.sqrt(2*c['layers']) if name in ('out','down') else 0.02)

    def forward(self, x, mask, cos, sin, cache=None, pos=0, capacity=0):
        p = self.p
        u = norm(x, p['nc'])
        history = x.new_zeros(x.size(0),2,x.size(-1)) if cache is None else cache[2]
        z = torch.cat((history,u),1)
        local = sum(z[:,2-i:2-i+x.size(1)] * p['conv'][i].to(x.dtype) for i in range(3))
        x = x + torch.sigmoid(u + p['gate'].to(x.dtype)) * local
        q, k, v = (a.view(x.size(0),x.size(1),self.h,-1).transpose(1,2)
                   for a in (norm(x,p['n1']) @ p['qkv']).chunk(3,-1))
        q, k = rope(q,cos,sin), rope(k,cos,sin)
        if cache is not None:
            cache[0][:,:,pos:pos+x.size(1)].copy_(k)
            cache[1][:,:,pos:pos+x.size(1)].copy_(v)
            cache[2].copy_(z[:,-2:])
            k, v = cache[0][:,:,:pos+x.size(1)], cache[1][:,:,:pos+x.size(1)]
        a = F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=0.)
        x = x + a.transpose(1,2).reshape_as(x) @ p['out']
        a, b = (norm(x,p['n2']) @ p['up']).chunk(2,-1)
        x = x + (F.silu(a)*b) @ p['down']
        if capacity:
            keys = k.new_zeros(k.size(0),self.h,capacity,k.size(-1)); values = torch.zeros_like(keys)
            keys[:,:,:k.size(2)].copy_(k); values[:,:,:v.size(2)].copy_(v)
            return x, (keys,values,z[:,-2:].clone())
        return x

class Model(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.embed = nn.Parameter(torch.empty(c['vocab'],c['width']))
        nn.init.normal_(self.embed,std=0.02)
        self.final = nn.Parameter(torch.ones(c['width']))
        self.blocks = nn.ModuleList([Block(c) for _ in range(c['layers'])])
        half = c['width']//c['heads']//2
        angle = torch.arange(c['prefix']+1024).float()[:,None] * 10000.**(-torch.arange(half).float()/half)
        self.register_buffer('cos', angle.cos()[None,None], persistent=False)
        self.register_buffer('sin', angle.sin()[None,None], persistent=False)

    def hidden(self, ids, prefix):
        t = ids.size(1)
        idx = torch.arange(t,device=ids.device)
        q, k, pre = idx[None,:,None], idx[None,None,:], prefix[:,None,None]
        mask = (((q<pre)&(k<pre)) | ((q>=pre)&(k<=q)))[:,None]
        x = F.embedding(ids,self.embed)
        if torch.is_autocast_enabled(): x = x.to(torch.get_autocast_dtype('cuda'))
        for block in self.blocks:
            args = (x,mask,self.cos[:,:,:t],self.sin[:,:,:t])
            x = checkpoint(block,*args,use_reentrant=False) if self.training else block(*args)
        return norm(x,self.final)

    def forward(self, rows):
        c = self.c
        ids, prefix, length = rows[:,:c['seq']], rows[:,-2], rows[:,-1]
        x = self.hidden(ids,prefix)
        offsets = torch.arange(c['target'],device=rows.device)[None]
        idx = prefix[:,None]-1+offsets
        states = x.gather(1,idx[:,:,None].expand(-1,-1,x.size(-1)))
        labels = ids.gather(1,idx+1)
        logits = states @ self.embed.T
        loss = F.cross_entropy(logits.float().flatten(0,1),labels.flatten(),reduction='none').view_as(labels)
        return ((loss*(offsets<length[:,None])).sum(1)/length).mean()

    def prefill(self, ids):
        t = ids.size(1); cache = []
        x = F.embedding(ids,self.embed)
        if torch.is_autocast_enabled(): x = x.to(torch.get_autocast_dtype('cuda'))
        for block in self.blocks:
            x, kv = block(x,None,self.cos[:,:,:t],self.sin[:,:,:t],capacity=t+1024)
            cache.append(kv)
        return (norm(x[:,-1],self.final) @ self.embed.T).float(), cache

    def decode(self, token, cache, pos):
        x = F.embedding(token[:,None],self.embed)
        if torch.is_autocast_enabled(): x = x.to(torch.get_autocast_dtype('cuda'))
        for block, kv in zip(self.blocks,cache):
            x = block(x,None,self.cos[:,:,pos:pos+1],self.sin[:,:,pos:pos+1],kv,pos)
        return (norm(x[:,0],self.final) @ self.embed.T).float()

with torch.device('meta'):
    num_params = sum(p.numel() for p in Model(C).parameters())
print(f'Parameters: {num_params:,} ({num_params/1e6:.1f}M)')

Parameters: 264,895,488 (264.9M)


## Two-GPU training

In [15]:
class Rows(Dataset):
    def __init__(self, cfg, rank, world):
        self.c, self.rank, self.world = cfg, rank, world
        self.data = np.memmap(Path(cfg['data'])/'train.bin',dtype=np.uint16,mode='r',
                              shape=(cfg['counts']['train'],cfg['seq']+2))
    def __len__(self): return 2**31
    def __getitem__(self, index):
        rng = np.random.default_rng(self.c['seed']+1+index*self.world+self.rank)
        ix = rng.integers(len(self.data),size=self.c['micro'])
        return torch.from_numpy(np.array(self.data[ix],dtype=np.int64))

@torch.no_grad()
def validate(model, valid, rank, world, device):
    model.eval(); stats = torch.zeros(2,device=device)
    for i in range(rank,min(len(valid),256),world):
        row = torch.from_numpy(np.array(valid[i:i+1],dtype=np.int64)).to(device)
        with torch.autocast('cuda',dtype=torch.float16): value = model(row)
        stats[0] += value; stats[1] += 1
    dist.all_reduce(stats)
    model.train()
    return (stats[0]/stats[1]).item()

def train_worker(rank, world, cfg):
    torch.set_num_threads(2); torch.cuda.set_device(rank)
    device = torch.device('cuda',rank)
    dist.init_process_group('nccl',rank=rank,world_size=world)
    torch.manual_seed(cfg['seed'])
    model = Model(cfg).to(device)
    groups = [dict(params=[p for n,p in model.named_parameters() if n.split('.')[-1] in ('embed','qkv','out','up','down')],weight_decay=cfg['weight_decay']),
              dict(params=[p for n,p in model.named_parameters() if n.split('.')[-1] not in ('embed','qkv','out','up','down')],weight_decay=0.)]
    opt = torch.optim.AdamW(groups,lr=cfg['lr'],betas=(0.9,0.95),eps=1e-8,fused=True)
    scaler = torch.amp.GradScaler('cuda',init_scale=1024.)
    step, total, best, history = 0, 100000, float('inf'), []
    out = Path(cfg['out'])
    if cfg['resume'] is not None:
        ck = torch.load(cfg['resume'],map_location='cpu',weights_only=False)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['optimizer']); scaler.load_state_dict(ck['scaler'])
        step, total, best, history = ck['step'], ck['total'], ck['best'], ck['history']
        if rank == 0:
            import shutil
            source = Path(cfg['resume']).parent/'best.pt'
            if source.resolve() != (out/'best.pt').resolve(): shutil.copyfile(source,out/'best.pt')
        del ck
    ddp = DDP(model,device_ids=[rank],broadcast_buffers=False,gradient_as_bucket_view=True)
    dataset = Rows(cfg,rank,world)
    loader = iter(DataLoader(dataset,batch_size=None,sampler=range(step*cfg['accum'],2**31),
                             num_workers=0,pin_memory=True))
    valid = np.memmap(Path(cfg['data'])/'val.bin',dtype=np.uint16,mode='r',
                      shape=(cfg['counts']['val'],cfg['seq']+2))
    def save(name, full=False):
        ck = dict(model=model.state_dict(),step=step,total=total,best=best,history=history,config=cfg)
        if full: ck.update(optimizer=opt.state_dict(),scaler=scaler.state_dict())
        torch.save(ck,out/(name+'.tmp')); os.replace(out/(name+'.tmp'),out/name)
    times = []; running_loss = torch.zeros((),device=device)
    control = torch.zeros(2,device=device,dtype=torch.float64)
    start_step = step
    while step < total:
        if step % 10 == 0 or step == start_step:
            if rank == 0: control[0] = time.monotonic() >= cfg['stop_at']
            dist.broadcast(control,0)
            if control[0].item(): break
        warm = min((step+1)/cfg['warmup'],1.)
        progress = min(max((step-cfg['warmup'])/max(total-cfg['warmup'],1),0.),1.)
        lr = cfg['lr']*warm*(0.1+0.9*0.5*(1+math.cos(math.pi*progress)))
        for group in opt.param_groups: group['lr'] = lr
        measuring = cfg['resume'] is None and step < 10
        if measuring: torch.cuda.synchronize()
        t = time.monotonic(); opt.zero_grad(set_to_none=True)
        for micro in range(cfg['accum']):
            rows = next(loader).to(device,non_blocking=True)
            sync = ddp.no_sync() if micro < cfg['accum']-1 else nullcontext()
            with sync:
                with torch.autocast('cuda',dtype=torch.float16): loss = ddp(rows)/cfg['accum']
                scaler.scale(loss).backward()
            running_loss += loss.detach()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
        scaler.step(opt); scaler.update(); step += 1
        if measuring:
            torch.cuda.synchronize(); times.append(time.monotonic()-t)
        if cfg['resume'] is None and step == 10:
            speed = torch.tensor(float(np.median(times[2:])),device=device)
            dist.all_reduce(speed,op=dist.ReduceOp.MAX)
            if rank == 0:
                control[1] = step+max(1,int((cfg['stop_at']-time.monotonic())*0.90/speed.item()))
                print(f'{speed.item():.2f}s/update | batch {world*cfg["micro"]*cfg["accum"]} | target {int(control[1].item()):,}',flush=True)
            dist.broadcast(control,0); total = int(control[1].item())
        if step % 100 == 0:
            dist.all_reduce(running_loss)
            if rank == 0: print(f'step {step:,}/{total:,} | train {running_loss.item()/(world*(step-start_step if step-start_step<100 else 100)):.4f}',flush=True)
            running_loss.zero_()
        if step % 500 == 0:
            val = validate(model,valid,rank,world,device)
            history.append(dict(step=step,val_loss=val))
            improved = val < best; best = min(best,val)
            if rank == 0:
                print(f'step {step:,} | val {val:.4f}',flush=True)
                if improved: save('best.pt')
                save('latest.pt',full=True)
                (out/'history.json').write_text(json.dumps(history))
            dist.barrier()
    val = validate(model,valid,rank,world,device)
    history.append(dict(step=step,val_loss=val))
    improved = val < best; best = min(best,val)
    if rank == 0:
        if improved: save('best.pt')
        save('latest.pt',full=True)
        (out/'history.json').write_text(json.dumps(history))
        print(f'Finished step {step:,} | val {val:.4f} | best {best:.4f}',flush=True)
    dist.barrier(); dist.destroy_process_group()

def launch(function, *args):
    payload = OUT/'worker.pkl'
    with payload.open('wb') as f: cloudpickle.dump((function,args),f)
    with socket.socket() as s:
        s.bind(('127.0.0.1',0)); port = s.getsockname()[1]
    command = "import cloudpickle,os; f=open(os.environ['WORKER_PAYLOAD'],'rb'); fn,args=cloudpickle.load(f); f.close(); fn(int(os.environ['RANK']),2,*args)"
    jobs = []
    for rank in range(2):
        env = dict(os.environ,MASTER_ADDR='127.0.0.1',MASTER_PORT=str(port),RANK=str(rank),
                   WORKER_PAYLOAD=str(payload),OMP_NUM_THREADS='2',TOKENIZERS_PARALLELISM='false')
        jobs.append(subprocess.Popen([sys.executable,'-u','-c',command],env=env))
    while any(p.poll() is None for p in jobs):
        if any(p.poll() not in (None,0) for p in jobs):
            for p in jobs:
                if p.poll() is None: p.terminate()
            raise RuntimeError('GPU worker failed; see its traceback above.')
        time.sleep(0.5)
    if any(p.returncode for p in jobs): raise RuntimeError('GPU worker failed; see its traceback above.')

cfg = dict(C,out=str(OUT),data=str(DATA),counts=counts,resume=RESUME,
           stop_at=START+(C['session_hours']-C['reserve_hours'])*3600)
launch(train_worker,cfg)

step 4,000/4,222 | train 2.3494
step 4,000 | val 2.3368


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)


step 4,100/4,222 | train 2.3229
step 4,200/4,222 | train 2.3259
Finished step 4,222 | val 2.3361 | best 2.3358


## Generation

In [16]:
@torch.inference_mode()
def generate(model, tokenizer, prompt, temperature=0., penalty=1.08, seed=781):
    c = model.c; device = model.embed.device
    pad, bos = tokenizer.token_to_id('<pad>'), tokenizer.token_to_id('<bos>')
    tokens = [bos]+tokenizer.encode(prompt.strip()).ids[-(c['prefix']-1):]
    ids = torch.tensor([tokens],device=device)
    rng = torch.Generator(device=device).manual_seed(seed)
    generated = torch.empty(1024,dtype=torch.long,device=device)
    seen = torch.zeros(c['vocab'],dtype=torch.bool,device=device)
    with torch.autocast('cuda',dtype=torch.float16):
        logits, cache = model.prefill(ids)
        for n in range(1024):
            scores = torch.where(seen,torch.where(logits[0]>0,logits[0]/penalty,logits[0]*penalty),logits[0])
            if n >= 4:
                grams = generated[:n].unfold(0,4,1)
                repeated = (grams[:,:3]==generated[n-3:n]).all(-1)
                bans = torch.zeros(c['vocab'],dtype=torch.int32,device=device)
                bans.scatter_reduce_(0,grams[:,3],repeated.int(),reduce='amax',include_self=True)
                scores.masked_fill_(bans.bool(),-1e9)
            scores[pad] = scores[bos] = -1e9
            scores[tokenizer.get_vocab_size():] = -1e9
            if temperature == 0: token = scores.argmax().view(1)
            else:
                top, ix = scores.topk(20)
                token = ix[torch.multinomial((top/temperature).softmax(-1),1,generator=rng)]
            generated[n] = token[0]; seen[token] = True
            logits = model.decode(token,cache,len(tokens)+n)
            if (n+1)%32 == 0:
                words = tokenizer.decode(generated[:n+1].tolist(),skip_special_tokens=True).split()
                if len(words) >= 129: return ' '.join(words[:128])
    return ' '.join(words[:128])

def generate_worker(rank, world, cfg, tasks, settings, tag):
    torch.set_num_threads(2); torch.cuda.set_device(rank)
    model = Model(cfg).to(rank)
    ck = torch.load(Path(cfg['out'])/'best.pt',map_location='cpu',weights_only=False)
    model.load_state_dict(ck['model']); del ck
    model.eval()
    tokenizer = Tokenizer.from_file(str(Path(cfg['out'])/'tokenizer.json'))
    result = {}
    for i in range(rank,len(tasks),world):
        uid, text = tasks[i]
        result[uid] = [generate(model,tokenizer,text,temp,penalty,cfg['seed']+i) for temp,penalty in settings]
    (Path(cfg['out'])/f'{tag}_{rank}.json').write_text(json.dumps(result,ensure_ascii=False))

## Decode selection

In [17]:
settings = [(0.,1.08), (0.65,1.08)]
tasks = [(str(i),d['prompt']) for i,d in enumerate(dev)]
launch(generate_worker,cfg,tasks,settings,'dev_outputs')
outputs = {}
for rank in range(2): outputs.update(json.loads((OUT/f'dev_outputs_{rank}.json').read_text()))
candidates = [[outputs[str(i)][j] for i in range(len(dev))] for j in range(len(settings))]
decode_results = []; choice = 0
from bert_score import BERTScorer
torch.set_num_threads(4)
scorer = BERTScorer(model_type='bert-base-uncased',num_layers=9,
                   device='cpu',batch_size=8,rescale_with_baseline=False)
refs = [d['reference'] for d in dev]
for setting, texts in zip(settings,candidates):
    _, _, f1 = scorer.score(texts,refs)
    score = float(f1.mean())
    decode_results.append(dict(temperature=setting[0],penalty=setting[1],bert_f1=score))
    print(f'Temperature {setting[0]} | local BERTScore F1 {score:.4f}')
choice = int(np.argmax([d['bert_f1'] for d in decode_results])); del scorer
TEMP, PENALTY = settings[choice]
(OUT/'decode_results.json').write_text(json.dumps(dict(settings=decode_results,chosen=[TEMP,PENALTY])))

Temperature 0.0 | local BERTScore F1 0.6228
Temperature 0.65 | local BERTScore F1 0.6172


178

In [18]:
test_path = hf_hub_download(REPO,'Phase_1_Pretraining/Test/test.json',repo_type='dataset')
with open(test_path) as f: test = json.load(f)
tasks = [(uid,row['text']) for uid,row in test.items()]
launch(generate_worker,cfg,tasks,[(TEMP,PENALTY)],'test_outputs')
outputs = {}
for rank in range(2): outputs.update(json.loads((OUT/f'test_outputs_{rank}.json').read_text()))
prediction = {uid: {'text': outputs[uid][0]} for uid in test}
with (OUT/'prediction.json').open('w') as f: json.dump(prediction,f,ensure_ascii=False)
with zipfile.ZipFile(OUT/'submission.zip','w',zipfile.ZIP_DEFLATED) as z:
    z.write(OUT/'prediction.json',arcname='prediction.json')
print(f"Saved {len(prediction)} continuations: {OUT/'submission.zip'}")

Saved 1000 continuations: /kaggle/working/p1_arch3/submission.zip


In [19]:
bundle = OUT / "p1_arch3_output_bundle.zip"

files = [
    OUT / "best.pt",
    OUT / "latest.pt",
    OUT / "tokenizer.json",
    OUT / "config.json",
    OUT / "data_meta.json",
    OUT / "dev.json",
    OUT / "decode_results.json",
    DATA / "train.bin",
    DATA / "val.bin",
    OUT / "prediction.json",
    OUT / "submission.zip",
]
files += sorted(OUT.glob("dev_outputs_*.json"))
files += sorted(OUT.glob("test_outputs_*.json"))

with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_STORED) as z:
    for path in files:
        if path.is_file():
            z.write(path, arcname=str(Path("p1_arch3") / path.relative_to(OUT)))

print(f"Bundle: {bundle}")
print(f"Included {sum(path.is_file() for path in files)} files | {bundle.stat().st_size / 1e9:.2f} GB")

Bundle: /kaggle/working/p1_arch3/p1_arch3_output_bundle.zip
Included 15 files | 4.48 GB


## Architecture and run settings